# Day 039 Project: Chart Dashboard

## What You're Building

A four-panel chart dashboard saved to `chart_dashboard.png` — bar, line, scatter, and histogram panels, each showing a different perspective on the retail sales dataset.

**Deliverable:** You run every cell top-to-bottom. The final checks pass. A file named `chart_dashboard.png` exists in the current directory.

## Project Requirements

1. Load `RETAIL_CSV` (provided) into a DataFrame with a `revenue` column
2. Build your bar chart: top products by revenue
3. Build your line chart: cumulative revenue over orders
4. Build your scatter chart: price vs revenue
5. Build your histogram: distribution of revenue values
6. Compose all four into a 2×2 dashboard and save as `chart_dashboard.png`
7. Verify with `_run_project_checks()`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

import matplotlib.pyplot as plt

def bar_chart(ax, labels, values, title='', xlabel='', ylabel=''):
    ax.bar(labels, values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45)
    return ax


import matplotlib.pyplot as plt

def line_chart(ax, x, y, title='', xlabel='', ylabel='', label=None):
    ax.plot(x, y, marker='o', label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if label:
        ax.legend()
    return ax


import matplotlib.pyplot as plt

def scatter_chart(ax, x, y, title='', xlabel='', ylabel=''):
    ax.scatter(x, y)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return ax


import matplotlib.pyplot as plt

def histogram(ax, data, bins=10, title='', xlabel=''):
    ax.hist(data, bins=bins, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    return ax


import os
import matplotlib.pyplot as plt

def multi_chart_figure(df, out_path='dashboard.png'):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Top-left: bar chart — revenue by product
    totals = df.groupby('product')['revenue'].sum().sort_values(ascending=False)
    bar_chart(axes[0, 0], totals.index.tolist(), totals.values.tolist(),
              'Revenue by Product', 'Product', 'Revenue ($)')

    # Top-right: line chart — cumulative revenue
    df_s = df.sort_values('order_id').reset_index(drop=True)
    line_chart(axes[0, 1], list(range(1, len(df_s) + 1)),
               df_s['revenue'].cumsum().tolist(),
               'Cumulative Revenue', 'Order #', 'Revenue ($)',
               label='cumulative')

    # Bottom-left: scatter — price vs revenue
    scatter_chart(axes[1, 0], df['price'].tolist(), df['revenue'].tolist(),
                  'Price vs Revenue', 'Price ($)', 'Revenue ($)')

    # Bottom-right: histogram — revenue distribution
    histogram(axes[1, 1], df['revenue'].tolist(), bins=8,
              title='Revenue Distribution', xlabel='Revenue ($)')

    fig.suptitle('Sales Dashboard', fontsize=16)
    plt.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', dpi=100)
    plt.close(fig)
    return out_path


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
df = pd.read_csv(io.StringIO(RETAIL_CSV))
df['revenue'] = df['price'] * df['quantity']
print(f'Loaded {len(df)} rows × {len(df.columns)} columns')

## Your Dashboard

In [ ]:
# TODO: Build your chart dashboard
# Option A — call multi_chart_figure directly:
# out = multi_chart_figure(df, 'chart_dashboard.png')
# print(f'Saved to {out}')

# Option B — build it manually for more control:
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# bar_chart(axes[0, 0], ...)
# line_chart(axes[0, 1], ...)
# scatter_chart(axes[1, 0], ...)
# histogram(axes[1, 1], ...)
# fig.suptitle('Sales Dashboard', fontsize=16)
# plt.tight_layout()
# fig.savefig('chart_dashboard.png', bbox_inches='tight', dpi=100)
# plt.close(fig)

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0
    _out = 'chart_dashboard.png'

    # Check 1: df has revenue column
    try:
        assert 'df' in globals() and 'revenue' in df.columns, \
            "'revenue' column missing"
        passed += 1; print(f'\u2705 Check 1: df loaded with revenue ({len(df)} rows)')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: chart_dashboard.png exists
    try:
        assert os.path.exists(_out), \
            f'{_out!r} not found — did you call fig.savefig({_out!r})?'
        passed += 1; print(f'\u2705 Check 2: {_out} exists')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: valid PNG
    try:
        with open(_out, 'rb') as f:
            _magic = f.read(4)
        assert _magic == b'\x89PNG'
        passed += 1; print('\u2705 Check 3: valid PNG file')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: file size > 10 KB (multi-panel content)
    try:
        _sz = os.path.getsize(_out)
        assert _sz > 10000, \
            f'file too small ({_sz} bytes) — multi-panel should exceed 10 KB'
        passed += 1; print(f'\u2705 Check 4: file size {_sz:,} bytes')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: no open figures (all closed)
    try:
        _open = plt.get_fignums()
        assert len(_open) == 0, \
            f'{len(_open)} figure(s) still open — call plt.close() after savefig'
        passed += 1; print('\u2705 Check 5: no lingering open figures')
    except Exception as e:
        plt.close('all')
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add colour: pass a `color=` or `cmap=` argument to bar/scatter
- Sort the bar chart by revenue descending before plotting
- Add a second line series to the line chart (e.g. separate Widget and Gadget revenue over time) by calling `line_chart` twice on the same ax with different labels
- Annotate the scatter plot: `ax.annotate('Gadget', xy=(150, 1050), ...)` to label the highest-revenue point
- Try `plt.style.use('seaborn-v0_8')` at the top for a cleaner look
- On Day 40 you will ask an LLM to narrate these charts — keep `df` handy